# Implementación del mecanismo de atención

Una vez se ha llevado a cabo la preparación y carga del corpus, tenemos
disponibles los datos de entrada del mecanismo de atención del LLM, los token
embeddings.

El objetivo del mecanismo de atención es, dado un conjunto de token embeddings
de entrada, calcular un context vector para cada uno de ellos.

En los contenidos del presente notebook:

1. Se detalla en qué consiste el mecanismo de atención de manera general
2. Se transforma el mecanismo de atención general en un mecanismo de atención
   causal
3. Se utilizan varios mecanismos de atención para implementar el mecanismo de
   atención causal multi-head de GPT-2

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(123)

## 1. Explicación del mecanismo de atención

El mecanismo de atención toma como entrada una serie de token embeddings y tiene
como objetivo calcular un context vector para cada uno de ellos.

De manera intuitiva, el resultado del mecanismo de atención, el context vector,
se puede entender de la siguiente manera:
Imaginemos que estamos calculando el context vector Zx para una palabra X en el
contexto de una oración. Zx recogerá, de forma numérica, el significado de la
palabra X dentro de su oración.

Con el objetivo de facilitar la comprensión del mecanismo de atención, aunque la
implementación final utilizará token embeddings con una alta dimensionalidad, se
tomarán como entrada unos token embeddings de ejemplo con una dimensionalidad
muy reducida, facilitando verificar los resultados obtenidos:

In [2]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Once
   [0.55, 0.87, 0.66], # upon  (x_2)
   [0.57, 0.85, 0.64], # a
   [0.22, 0.58, 0.33], # time
   [0.77, 0.25, 0.10], # there
   [0.05, 0.80, 0.55]] # was
)

Igualmente, aunque el objetivo final es calcular todos los context vectors, con
el objetivo de ilustrar el funcionamiento del mecanismo de atención de una forma
sencilla, se calcula el context vector (`z_2`) del token embedding `x_2` antes
de pasar a generalizar la solución.

In [3]:
x_2 = inputs[1]

El cálculo del context vector `z_2` se lleva a cabo en cuatro pasos que se
describen a continuación.

### Paso 1: Inicialización de las matrices entrenables

El mecanismo de atención cuenta con 3 matrices de pesos que se irán actualizando
durante el entrenamiento:

- Query (`W_query`)
- Key (`W_key`)
- Value (`W_value`)

Los términos "query", "key" y "value" se toman del mundo de las bases de datos y
se adaptan al mecanismo de atención

La query representa el elemento actual que el modelo está intentando entender,
en nuestro ejemplo, el segundo elemento.

Como el las bases de datos, la key se utiliza para buscar. En el mecanismo de
atención la query se compara contra todas las keys de entrada, incluida la del
propio elemento a comprobar (`x_2`).

Finalmente, el value es el valor asociado a cada elemento de entrada.

Una vez la idea intuitiva detrás de estas matrices está aclarada, pasamos a
inicializarlas. Para hacer los cálculos más evidentes, se usan dimensiones de
entrada y salida (`d_in` y `d_out`) distintas, aunque en el LLM final ambas
dimensiones tendrán el mismo valor:

In [4]:
# La dimensión de entrada es la de los token embeddings de entrada (3)
d_in = inputs.shape[1]

# Se usa una dimensión de salida diferente para hacer los cálculos evidentes
d_out = 2

# En el modelo final requires_grad debe ser True, pero se inicializa a False
# para que los cálculos sean más sencillos de seguir
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

### Paso 2: Calcular el attention score (ω)

Para calcula el attention score, en primer lugar necesitamos calcular el vector
query del token embedding a examinar (`x_2`) proyectándolo sobre la matriz
`W_query`:

In [5]:
query_2 = x_2 @ W_query
print(query_2)

tensor([0.4306, 1.4551])


A continuación, calculamos el producto escalar del vector query (`query_2`) con
cada uno de los keys proyectados sobre la matriz `W_key`, obteniendo los
attention scores:

In [6]:
for index, x in enumerate(inputs):
    key_vector = x @ W_key
    dot_product = query_2.dot(key_vector)
    print(f"Attention score para x_2 con la entrada {index}: {dot_product:.4f}")

Attention score para x_2 con la entrada 0: 1.2705
Attention score para x_2 con la entrada 1: 1.8524
Attention score para x_2 con la entrada 2: 1.8111
Attention score para x_2 con la entrada 3: 1.0795
Attention score para x_2 con la entrada 4: 0.5577
Attention score para x_2 con la entrada 5: 1.5440


Sin embargo, calcular el producto escalar de esta manera es ineficiente.
Se pueden obtener los attention scores mediante multiplicación de matrices:

In [7]:
keys = inputs @ W_key
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


### Paso 3: Calcular los attention weights (α)

Los attention weights no son más que los attention scores normalizados. En el
caso de GPT-2 la función de normalización utilizada es `softmax`.

Nótese que GPT-2 divide los attencion scores antes de normalizarlos entre la
raíz cuadrada de de la dimensión de los embeddings. La razón es que esto mejora
el rendimiento del entrenamiento porque evita gradientes pequeños que podrían
llevar a un problema de vanishing gradient.

In [8]:
d_k = keys.shape[-1]
print(d_k)

2


In [9]:
# Matemáticamente, la raíz cuadrada es equivalente a elevar a 0.5
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


Como se puede comprobar, la suma de los attention weights es 1:

In [10]:
from functools import reduce

reduce(lambda x, y: x + y, attn_weights_2)

tensor(1.)

### Paso 4: Calcular el context vector (z)

Finalmente, se calcula el context vector, la salida del mecanismo de atención.

Dicho vector es la suma suma ponderada por peso (α) de los vectores de valores:

In [11]:
z_2 = torch.zeros(d_out)

for index, x in enumerate(inputs):
    value_vector = x @ W_value
    attn_weight = attn_weights_2[index]

    mul = attn_weight * value_vector
    z_2 += mul

print(z_2)

tensor([0.3061, 0.8210])


Una vez más, este cálculo se puede hacer de manera eficiente mediate
multiplicación de matrices:

In [12]:
values = inputs @ W_value
z_2 = attn_weights_2 @ values
print(z_2)

tensor([0.3061, 0.8210])


### Obteniendo todos los context vectors

Hasta ahora se ha calculado un único context vector (`z_2`) para explicar de
manera sencilla el mecanismo de atención. Sin embargo, un mecanismo de atención
real debe calcular el context vector de todas las entradas.

Una vez el concepto está claro, la implementación es relativamente sencilla:

In [13]:
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

In [14]:
torch.manual_seed(123)
sa = SelfAttention(d_in, d_out)
print(sa(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


Como se puede observar, el context vector z_2 obtenido es idéntico al calculado
anteriormente.

## 2. Atención causal

GPT-2 utiliza un mecanismo de atención causal multi-head.

Que el mecanismo de atención sea causal quiere decir que el modelo presta
atención exclusivamente a las palabras pasadas y presentes. Las palabras futuras
son enmascaradas para ocultárselas al modelo, que, en última instancia, deberá
predecirlas.

Para ello calculamos los attention weights (α) como se ha detallado
anteriormente:

In [15]:
queries = inputs @ W_query
keys = inputs @ W_key

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

print(attn_weights)

tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])


Y los enmascaramos multiplicándolos por una matriz con unos desde su diagonal
hacia abajo y ceros en los espacios restantes:

In [16]:
context_length = attn_scores.shape[0]
mask = torch.tril(torch.ones(context_length, context_length))
print(mask)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [17]:
masked_attn_weights = attn_weights * mask
print(masked_attn_weights)

tensor([[0.1551, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1500, 0.2264, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1503, 0.2256, 0.2192, 0.0000, 0.0000, 0.0000],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.0000, 0.0000],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.0000],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])


Finalmente, normalizamos los valores obtenidos de manera que cada fila sume uno:

In [18]:
row_sums = masked_attn_weights.sum(dim=-1, keepdim=True)
masked_attn_weights_norm = masked_attn_weights / row_sums
print(masked_attn_weights_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3986, 0.6014, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2526, 0.3791, 0.3683, 0.0000, 0.0000, 0.0000],
        [0.2265, 0.2839, 0.2794, 0.2103, 0.0000, 0.0000],
        [0.1952, 0.2363, 0.2331, 0.1820, 0.1534, 0.0000],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])


La atención causal se implementa de manera genérica en la siguiente clase que
añade además un mecanismo para implementar dropout.

En un LLM la técnica de dropout funciona de manera similar a la empleada en
redes neuronales salvo porque, en lugar de apagar neuronas de manera aleatoria,
se enmascaran pesos aleatorios, lo cual ayuda a evitar problemas de overfitting.

In [19]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout):
        super().__init__()

        self.d_out = d_out
        
        # En lugar de utilizar nn.Parameter como en la clase SelfAttention se
        # utiliza nn.Linear. Cuando su bias está desactivado también es
        # equivalente a la multiplicación de matrices, pero generalmente mejora
        # el entrenamiento
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # La entrada ahora son batches, por lo que la variable context_length es
        context_length = x.shape[1]
        
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.transpose(1, 2)        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        mask = torch.tril(torch.ones(context_length, context_length))
        masked_attn_weights = attn_weights * mask

        row_sums = masked_attn_weights.sum(dim=-1, keepdim=True)
        masked_attn_weights_norm = masked_attn_weights / row_sums

        attn_weights = self.dropout(masked_attn_weights_norm)

        context_vec = attn_weights @ values
        return context_vec

In [20]:
# Creamos un batch consistente en los datos de entrada duplicados
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
print(ca(batch))

torch.Size([2, 6, 3])
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)


## 3. Atención multi-head

Como ya se ha mencionado anteriormente, GPT-2 utiliza un mecanismo de atención
causal multi-head.

Que el mecanismo de atención sea multi-head quiere decir que se
aplica el proceso de atención tantas veces como heads tenga el modelo y se
combinan los context vectors resultantes para obtener el resultado final.

La implementación es tan sencilla como usar la clase `CausalAttention` tantas
veces como heads queramos implementar y concatenar los resultados. Por ejemplo,
con dos heads el resultado sería:

In [21]:
torch.manual_seed(123)
context_length = batch.shape[1]

head1 = CausalAttention(d_in, d_out, context_length, 0.0)
head2 = CausalAttention(d_in, d_out, context_length, 0.0)

torch.cat([head1(batch), head2(batch)], dim=-1)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)

Si bien esta implementación es tremendamente sencilla de entender, en un LLM
real habría que agrupar las multiplicaciones de matrices para optimizar los
cálculos.